# KX2 continuous motion

Complete [hello world](hello-world.ipynb) through setup first. These cells assume a connected `kx2`.

Streaming sends CANopen interpolation points at 8 ms intervals. Setup configures the drive buffers and synchronized motion channels.

## Capture a starting pose

All locations are absolute grip-center coordinates in millimeters, with the KX2 yaw convention described in hello world.

In [ ]:
from pylabrobot.paa.kx2 import CartesianPose
from pylabrobot.resources import Coordinate, Rotation

start = await kx2.request_gripper_pose()

## Linear Cartesian path

After verifying a clear 5 mm vertical segment, run this move. Both speed and acceleration must be provided. The planner checks joint travel at the sampled poses; it does not check collisions.

In [ ]:
target = Coordinate(start.location.x, start.location.y, start.location.z + 5)
await kx2.move_to_location(target, start.rotation.z, path="linear",
                          max_gripper_speed=5, max_gripper_acceleration=20)

## Rotate at a fixed grip center

A linear path can change yaw while keeping the grip center in place. Set the tool offsets for your gripper and verify clearance for the rotation. Angular speed and acceleration use degrees per second and degrees per second squared; their defaults are 30 and 100.

In [ ]:
current = await kx2.request_gripper_pose()
await kx2.move_to_location(
  current.location, current.rotation.z + 1, path="linear",
  max_gripper_speed=5, max_gripper_acceleration=20,
  max_gripper_angular_speed=1, max_gripper_angular_acceleration=2,
)

## Smooth waypoints

`move_through_waypoints` fits a smooth spline, so the curve can overshoot the line segments joining your points. Check the whole curve. Keep the first waypoint at the measured current pose.

In [ ]:
current = await kx2.request_gripper_pose()
waypoints = [current, CartesianPose(
  Coordinate(current.location.x, current.location.y, current.location.z + 5), current.rotation
)]
await kx2.move_through_waypoints(waypoints, speed=5, acceleration=20)

## Parametric paths

`move_parametric(path_fn, duration)` samples absolute poses at times in seconds. It does not impose a speed or acceleration cap. The path must start at the measured pose and remain continuous and within travel and motion limits; this holds the measured pose without a displacement.

In [ ]:
held = await kx2.request_gripper_pose()

def hold_pose(t: float) -> CartesianPose:
  """Return the same absolute pose throughout the requested interval."""
  return held

await kx2.move_parametric(hold_pose, duration=1)

## Disconnect

In [ ]:
await kx2.stop()